# Supply Chain Data Segmentation
**Kyle R. Hudson — Northwest Missouri State University**
MS Data Analytics Capstone Project

# Warehouse Assignment — Predictive Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

df = pd.read_csv('Dataset/Customer Shopping Trends/shopping_trends_project.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Warehouse Region Assignment
Six warehouses based on real locations. Each state is assigned to its nearest warehouse.

In [ ]:
warehouse_map = {
    # Lewisberry, PA — Northeast / Mid-Atlantic
    'Maine': 'Lewisberry_PA',
    'New Hampshire': 'Lewisberry_PA',
    'Vermont': 'Lewisberry_PA',
    'Massachusetts': 'Lewisberry_PA',
    'Rhode Island': 'Lewisberry_PA',
    'Connecticut': 'Lewisberry_PA',
    'New York': 'Lewisberry_PA',
    'New Jersey': 'Lewisberry_PA',
    'Pennsylvania': 'Lewisberry_PA',
    'Delaware': 'Lewisberry_PA',
    'Maryland': 'Lewisberry_PA',
    'West Virginia': 'Lewisberry_PA',
    'Virginia': 'Lewisberry_PA',

    # Marietta, GA — Southeast
    'North Carolina': 'Marietta_GA',
    'South Carolina': 'Marietta_GA',
    'Georgia': 'Marietta_GA',
    'Florida': 'Marietta_GA',
    'Alabama': 'Marietta_GA',
    'Mississippi': 'Marietta_GA',
    'Tennessee': 'Marietta_GA',
    'Kentucky': 'Marietta_GA',

    # Irving, TX — South / Southwest
    'Texas': 'Irving_TX',
    'Oklahoma': 'Irving_TX',
    'Arkansas': 'Irving_TX',
    'Louisiana': 'Irving_TX',
    'New Mexico': 'Irving_TX',
    'Arizona': 'Irving_TX',

    # Lee's Summit, MO — Midwest
    'Missouri': 'Lees_Summit_MO',
    'Iowa': 'Lees_Summit_MO',
    'Illinois': 'Lees_Summit_MO',
    'Indiana': 'Lees_Summit_MO',
    'Ohio': 'Lees_Summit_MO',
    'Michigan': 'Lees_Summit_MO',
    'Wisconsin': 'Lees_Summit_MO',
    'Minnesota': 'Lees_Summit_MO',
    'North Dakota': 'Lees_Summit_MO',
    'South Dakota': 'Lees_Summit_MO',
    'Nebraska': 'Lees_Summit_MO',
    'Kansas': 'Lees_Summit_MO',

    # Reno, NV — Mountain / Intermountain West
    'Nevada': 'Reno_NV',
    'Utah': 'Reno_NV',
    'Colorado': 'Reno_NV',
    'Wyoming': 'Reno_NV',
    'Montana': 'Reno_NV',
    'Idaho': 'Reno_NV',

    # Union City, CA — West Coast
    'California': 'Union_City_CA',
    'Oregon': 'Union_City_CA',
    'Washington': 'Union_City_CA',
    'Alaska': 'Union_City_CA',
    'Hawaii': 'Union_City_CA',
}

df['Warehouse'] = df['Location'].map(warehouse_map)
print('Warehouse assignment counts:')
print(df['Warehouse'].value_counts())
print(f'\nUnmapped states: {df["Warehouse"].isnull().sum()}')

## 2. Feature Engineering & Encoding

In [ ]:
features = ['Location', 'Category', 'Shipping Type', 'Season',
            'Gender', 'Age', 'Purchase Amount (USD)',
            'Frequency of Purchases', 'Previous Purchases']

X = df[features].copy()
y = df['Warehouse']

# Encode categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

print('Feature matrix shape:', X.shape)
X.head()

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} records')
print(f'Test set:     {X_test.shape[0]} records')

## 4. Model 1 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Random Forest Accuracy:', round(accuracy_score(y_test, y_pred_rf), 4))
print()
print(classification_report(y_test, y_pred_rf))

## 5. Model 2 — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('Logistic Regression Accuracy:', round(accuracy_score(y_test, y_pred_lr), 4))
print()
print(classification_report(y_test, y_pred_lr))

## 6. Confusion Matrix — Random Forest

In [ ]:
warehouses = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred_rf, labels=warehouses)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=warehouses, yticklabels=warehouses, ax=ax)
ax.set_title('Confusion Matrix — Random Forest')
ax.set_xlabel('Predicted Warehouse')
ax.set_ylabel('Actual Warehouse')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Mod5/confusion_matrix_rf.png', dpi=150)
plt.show()

## 7. Feature Importance — Random Forest

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot(kind='bar', ax=ax)
ax.set_title('Feature Importance — Random Forest')
ax.set_ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Mod5/feature_importance_rf.png', dpi=150)
plt.show()

## 8. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_rf), 4),
        round(accuracy_score(y_test, y_pred_lr), 4)
    ]
})
print(results.to_string(index=False))